In [41]:
import pandas as pd

In [ ]:
from datasets import load_dataset
ds = load_dataset("AgentPublic/cnil")

c:\Users\alixh\Documents\Projets\cnil-graphrag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\alixh\Documents\Projets\cnil-graphrag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alixh\.cache\huggingface\hub\datasets--AgentPublic--cnil. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administr

# Etape 1 : Exploration de la data

In [8]:
print(ds)
df = ds["train"].to_pandas()
print(df.head(5))
print(df["text"][0])

DatasetDict({
    train: Dataset({
        features: ['chunk_id', 'doc_id', 'chunk_index', 'chunk_xxh64', 'nature', 'status', 'nature_delib', 'title', 'full_title', 'number', 'date', 'text', 'chunk_text', 'embeddings_bge-m3'],
        num_rows: 64667
    })
})
                 chunk_id                doc_id  chunk_index  \
0  CNILTEXT000017651752_1  CNILTEXT000017651752            1   
1  CNILTEXT000017651752_2  CNILTEXT000017651752            2   
2  CNILTEXT000017651752_3  CNILTEXT000017651752            3   
3  CNILTEXT000017651752_4  CNILTEXT000017651752            4   
4  CNILTEXT000017651752_5  CNILTEXT000017651752            5   

        chunk_xxh64        nature   status        nature_delib  \
0  49574288592aa3f7  DELIBERATION  VIGUEUR  Autre autorisation   
1  f7b2925dd0cb489e  DELIBERATION  VIGUEUR  Autre autorisation   
2  656d16a4f33193a8  DELIBERATION  VIGUEUR  Autre autorisation   
3  683b5018adea0ad7  DELIBERATION  VIGUEUR  Autre autorisation   
4  6c6e72bc66fddc96  DEL

In [54]:
# Combien de décisions
print("df[doc_id].nunique() : ", df["doc_id"].nunique())

df[doc_id].nunique() :  9983


In [55]:
# Nature des sanctions 
print("sanctions :", df["nature"].value_counts())
print("sanctions :", df["nature_delib"].value_counts())

sanctions : nature
DELIBERATION    59768
DECISION         4899
Name: count, dtype: int64
sanctions : nature_delib
Autre autorisation                   23529
Avis                                 21909
Sanction                              6391
Référentiel/Règlement type/Norme      3738
Autorisation de recherche             2717
Autorisation de transfert             2020
Mise en demeure                       1145
Recommandation/Lignes directrices     1045
Disposition interne CNIL               951
Certification/Label                    651
Autorisation d'évaluation              138
Clôture                                118
Autre décision                          26
AVIS                                     2
Name: count, dtype: int64


In [56]:
# Types 
df.dtypes

chunk_id                        str
doc_id                          str
chunk_index                   int32
chunk_xxh64                     str
nature                          str
status                          str
nature_delib                    str
title                           str
full_title                      str
number                          str
date                 datetime64[us]
text                            str
chunk_text                      str
embeddings_bge-m3               str
dtype: object

In [ ]:

# Convert date to datetime 
df["date"] = pd.to_datetime(df["date"])

# We will only consider decisions with a sanction or a mise en demeure (nature_delib) for the graph
df_sanctions = df[(df["nature_delib"].isin(["Mise en demeure","Sanction"])) & (df["date"] >= "2018-05-25")].copy()
print("Number of sanctions and mise en demeure:", df_sanctions["doc_id"].nunique())

# Min and max date
print("Minimal date :", df_sanctions["date"].min())
print("Maximal date :", df_sanctions["date"].max())

# Adding the column year to the dataframe
df_sanctions["year"] = df_sanctions["date"].dt.year

# Number of sanctions and mise en demeure per year
df_sanctions_year = df_sanctions.groupby(["year","nature_delib"])["doc_id"].nunique().reset_index(name="nb_decisions")
print("Number of decisions per year and type :")
print(df_sanctions_year)

# Rules (recommendations and standards) for the vector search
df_rules = df[df["nature_delib"].isin(["Recommandation/Lignes directrices","Référentiel/Règlement type/Norme"]) & (df["date"] >= "2018-05-25")].copy()
print("Number of rules:", df_rules["doc_id"].nunique())

# Empty value


Number of sanctions and mise en demeure: 160
Minimal date : 2018-06-21 00:00:00
Maximal date : 2026-11-12 00:00:00
Number of decisions per year and type :
    year     nature_delib  nb_decisions
0   2018  Mise en demeure            22
1   2018         Sanction             7
2   2019  Mise en demeure             6
3   2019         Sanction             5
4   2020  Mise en demeure             4
5   2020         Sanction            12
6   2021  Mise en demeure             4
7   2021         Sanction            14
8   2022  Mise en demeure            24
9   2022         Sanction            16
10  2023  Mise en demeure             1
11  2023         Sanction            18
12  2024         Sanction             8
13  2025         Sanction            10
14  2026         Sanction             9
Number of rules: 84
Number of rules with empty text: 0


# Etape 2 : Extraction des décisions 

Extraction des sanctions et des mises en demeure pour le test de la recherche vectorielle
